In [11]:
# 1. Imports & Setup Library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Konfigurasi Tampilan Visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

# 2. Membuat Dataset Simulasi Kesehatan Jantung (Pengganti Database SQL untuk Colab)
np.random.seed(42)
n_patients = 300

df_patients = pd.DataFrame({
    'patient_id': range(1001, 1001 + n_patients),
    'age': np.random.randint(29, 78, size=n_patients),
    'sex': np.random.choice([0, 1], size=n_patients, p=[0.35, 0.65]), # 0: Wanita, 1: Pria
    'chest_pain_type': np.random.choice([0, 1, 2, 3], size=n_patients),
    'resting_bp': np.random.randint(94, 200, size=n_patients), # Tekanan darah sistolik
    'cholesterol': np.random.randint(126, 400, size=n_patients), # Kolesterol (mg/dl)
    'fasting_bs': np.random.choice([0, 1], size=n_patients, p=[0.85, 0.15]), # Gula darah puasa > 120 mg/dl
    'max_heart_rate': np.random.randint(71, 202, size=n_patients),
    'heart_disease': np.random.choice([0, 1], size=n_patients, p=[0.45, 0.55]) # Target (0: Sehat, 1: Berisiko)
})

# Ubah nilai kolom sex agar lebih mudah dibaca (Male/Female)
df_patients['gender'] = df_patients['sex'].apply(lambda x: 'Male' if x == 1 else 'Female')

# Tampilkan 5 data teratas
print("Dataset berhasil dibuat!")
df_patients.head()

Dataset berhasil dibuat!


,patient_id,age,sex,chest_pain_type,resting_bp,cholesterol,fasting_bs,max_heart_rate,heart_disease,gender
0,1001,67,1,3,176,142,0,184,1,Male
1,1002,57,0,3,135,253,0,148,1,Female
2,1003,43,0,0,134,376,0,200,1,Female
3,1004,71,1,0,194,304,0,107,0,Male
4,1005,36,0,1,99,307,0,90,1,Female


In [12]:
# 1. Ringkasan Statistik Medis Pasien
print("=== RINGKASAN STATISTIK DATA KESEHATAN ===")
stats_df = df_patients[['age', 'resting_bp', 'cholesterol', 'max_heart_rate']].describe().T
stats_df['range'] = stats_df['max'] - stats_df['min']
display(stats_df[['mean', 'std', 'min', '50%', 'max', 'range']])

# 2. Visualisasi Grafik EDA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafik A: Distribusi Kolesterol berdasarkan Status Penyakit Jantung
sns.histplot(data=df_patients, x='cholesterol', hue='heart_disease', kde=True, ax=axes[0], palette='Set2')
axes[0].set_title('Distribusi Kolesterol Pasien (0: Sehat, 1: Berisiko)')
axes[0].set_xlabel('Kolesterol (mg/dl)')
axes[0].set_ylabel('Jumlah Pasien')

# Grafik B: Boxplot Tekanan Darah berdasarkan Gender & Status Penyakit
sns.boxplot(data=df_patients, x='gender', y='resting_bp', hue='heart_disease', ax=axes[1], palette='Set1')
axes[1].set_title('Tekanan Darah Istirahat berdasarkan Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Tekanan Darah Istirahat (mmHg)')

plt.tight_layout()
plt.show()

=== RINGKASAN STATISTIK DATA KESEHATAN ===


,mean,std,min,50%,max,range
age,52.860000,14.367387,29.0,53.0,77.0,48.0
resting_bp,143.946667,30.771417,94.0,144.0,199.0,105.0
cholesterol,264.170000,80.588519,127.0,272.5,399.0,272.0
max_heart_rate,134.820000,38.317294,71.0,136.0,201.0,130.0


In [13]:
# 1. Uji Hipotesis Statistik (Uji t-Student)
# H0: Tidak ada perbedaan rata-rata kolesterol antara pasien sehat dan pasien berisiko penyakit jantung
# H1: Pasien berisiko penyakit jantung memiliki rata-rata kolesterol lebih tinggi

chol_healthy = df_patients[df_patients['heart_disease'] == 0]['cholesterol']
chol_diseased = df_patients[df_patients['heart_disease'] == 1]['cholesterol']

t_stat, p_val = stats.ttest_ind(chol_diseased, chol_healthy, alternative='greater')

print("=== HASIL UJI HIPOTESIS (T-TEST) ===")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")

if p_val < 0.05:
    print("Kesimpulan: Tolak H0. Terdapat bukti signifikan secara statistik bahwa rata-rata kolesterol pasien berisiko penyakit jantung lebih tinggi.\n")
else:
    print("Kesimpulan: Gagal Tolak H0. Tidak cukup bukti statistik.\n")


# 2. Feature Engineering (Persiapan Data untuk Machine Learning)
df_feat = df_patients.copy()

# A. One-Hot Encoding untuk kolom gender (mengubah teks menjadi angka biner)
df_feat = pd.get_dummies(df_feat, columns=['gender'], drop_first=True)

# B. Log Transformation untuk menstabilkan varians pada data kolesterol
df_feat['log_cholesterol'] = np.log1p(df_feat['cholesterol'])

# C. Interaction Feature: Membuat fitur baru rasio Tekanan Darah terhadap Detak Jantung Maksimum
df_feat['bp_hr_ratio'] = df_feat['resting_bp'] / df_feat['max_heart_rate']

print("=== HASIL FEATURE ENGINEERING (5 Data Teratas) ===")
display(df_feat[['patient_id', 'age', 'cholesterol', 'log_cholesterol', 'bp_hr_ratio', 'heart_disease']].head())

=== HASIL UJI HIPOTESIS (T-TEST) ===
t-statistic: -0.0786
p-value: 0.5313
Kesimpulan: Gagal Tolak H0. Tidak cukup bukti statistik.

=== HASIL FEATURE ENGINEERING (5 Data Teratas) ===


,patient_id,age,cholesterol,log_cholesterol,bp_hr_ratio,heart_disease
0,1001,67,142,4.962845,0.956522,1
1,1002,57,253,5.537334,0.912162,1
2,1003,43,376,5.932245,0.670000,1
3,1004,71,304,5.720312,1.813084,0
4,1005,36,307,5.730100,1.100000,1
